# Track 2 — Cluster efficiency: starting point

The fetch layer is done. This notebook gets you from zero to one real chart and one
real drill-down, then gets out of the way.

**Your task:** the CFO must cut GPU spend 20% next quarter without slowing research
down. Build the view that says where to cut, and what it costs if she's wrong.


In [ ]:
import pandas as pd, matplotlib.pyplot as plt
from mgai_client import MGAI

mg = MGAI()
mg.health()

## 1. Where the money went

`kind: "fact"` means this is deterministic and you can recompute it from the raw
data yourself. Always read `provenance.caveat` — it tells you what the number does
not mean.

In [ ]:
e = mg.efficiency_summary()
w = mg.rows(e)
print(e["provenance"]["caveat"], "\n")

price = mg.price_book()["usd_per_gpu_hour"]
w["usd"] = w.gpu_hours * price
w

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.barh(w.label[::-1], w.gpu_hours[::-1], color=["#1b4965", "#5fa8d3", "#bee9e8"][::-1])
for i, r in enumerate(w[::-1].itertuples()):
    ax.text(r.gpu_hours, i, f"  {r.gpu_hours:,.0f} GPU-h  (${r.usd:,.0f})", va="center")
ax.set_xlim(0, w.gpu_hours.max() * 1.45)
ax.set_title("Allocated → computed → completed", loc="left", weight="bold")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()

lost = 1 - w.iloc[-1].gpu_hours / w.iloc[0].gpu_hours
print(f"{lost:.0%} of allocated GPU time did not turn into completed work.")

**Before you put that number on a slide** — it stacks two different discounts
(idle-while-allocated, and did-not-complete), so a job that was both gets counted
twice. And a third of it is user cancellation, which is often good practice rather
than waste. Look at `/v1/waste/breakdown` and decide for yourself which rows are
actually recoverable.

In [ ]:
mg.rows(mg.waste_breakdown())

## 2. Don't trust the rankings

`kind: "judgment"` means a model produced this. It has a `confidence` and it can be
wrong. Here is the ranked list of underperforming nodes:

In [ ]:
u = mg.underperforming(entity_type="node", limit=8)
print("confidence:", u["confidence"], "|", u["provenance"]["caveat"], "\n")
mg.rows(u)

Those look like eight machines with eight problems. **Check before you
recommend draining anything.** Pull the findings for the top-ranked node and follow
the evidence:

In [ ]:
top = mg.rows(u).iloc[0]
f = mg.findings_df(resource_id=top.resource_id)
print(f"{len(f)} findings on {top.entity_id}\n")
f.groupby(["detectorId", "metadata_owner"]).size().sort_values(ascending=False).head()

Who owns those failures is where to **start**, not where to stop. One person owning
most of a machine's failures can mean their code is broken — or that a broken machine
is where their jobs happened to land. "Whoever owns the most failures" gets the one
real hardware fault on this cluster wrong. Compare that person against everyone
*else* on the same machine at the same time before you decide.

Now try the other direction — take a finding that has a `rootCauses` link and ask
the causal endpoint what is actually underneath it:

In [ ]:
fs = mg.findings(detector_id="rules::filesystem-latency-degraded", limit=1)[0]
for c in mg.causal(fs["id"]):
    print(c["root_cause"])
    print(f"confidence {c['confidence']}  agreement {c['algorithm_agreement']}\n")
    for cu in c["culprit"][:5]:
        print(f"  {cu['score']:.2f}  {cu['node']:24} {cu['type']}")

In [ ]:
# Arrays resolve too. Each machine's score is its share of the failures -- small and
# spread out, which is the evidence that the machines are NOT the cause.
at = mg.findings(detector_id="rules::array-task-failure", limit=1)[0]
for c in mg.causal(at["id"]):
    print(c["root_cause"])
    for cu in c["culprit"][:4]:
        print(f"  {cu['score']:.2f}  {cu['node']:24} {cu['type']}")

## 3. Totals are not as easy as they look

Findings carry structured impact. `impact_kind` and `impact_scope` are not
decoration — adding across them produces a number that means nothing.

- `lost` — work destroyed, rerunning costs that much again
- `degraded` — work finished, just slowly
- `consumed` — work finished, wastefully
- `unused_capacity` — a reserved resource never used; not compute at all

and `impact_scope` says whether a row covers one job, one node, or one person's
entire four months.

In [ ]:
a = mg.findings_df()
(a.groupby(["metadata_impact_kind", "metadata_impact_scope"])
   .metadata_impact_gpu_hours.agg(["size", "sum"]).round(0))

Findings also overlap — an idle interactive session is usually also a job that
never computed, so it trips two rules. Deduplicate by job before totalling anything.
`docs/rules.md` names the largest overlaps, and every rule's exact threshold.

## 4. Over to you

Three tiles are required: **where the money goes**, **where to cut**, and **what it
costs if you're wrong**. Everything else is open.

You're scored on whether a non-engineer knows what to do in 30 seconds, whether
every number is in dollars / hours / percent of capacity, whether recommendations
state their confidence, and whether you can click from a business number down to
the evidence.

Endpoints not touched above: `mg.queue_latency()`, `mg.scaling_efficiency()`,
`mg.recommendations()`, `mg.neighbor()`. Full spec at
[localhost:8000/docs](http://localhost:8000/docs).

`GET /v1/policies/rules` lists every rule, including the one that is armed and has
never fired — that silence is a result too.